# 🚀 多智能体系统与工作流模式

**欢迎来到 Kaggle 5 天 Agents 课程！**

在上一份 Notebook 中，你构建了一个能够执行动作的**单智能体**。现在，你将学习如何通过构建**智能体团队**来扩展能力。

就像现实中的团队一样，你可以创建多个分工明确的专用 Agent 协作解决复杂问题。这就是**多智能体系统（multi-agent system）**，也是 AI Agent 开发中最强大的核心概念之一。

在本 Notebook 中，你将：

- ✅ 学习在 [Agent Development Kit (ADK)](https://google.github.io/adk-docs/) 中何时使用多智能体系统
- ✅ 用 LLM 作为“管理者”搭建你的第一个系统
- ✅ 学习三种核心工作流模式（Sequential、Parallel、Loop）来协调你的 Agent 团队

## ⚙️ 第 1 部分：环境准备

### 1.1：安装依赖

Kaggle Notebooks 环境已经预装了 Python 版 [google-adk](https://google.github.io/adk-docs/) 及其所需依赖，因此你在本 Notebook 中无需额外安装包。

如果你要在课程外、自己的 Python 开发环境中安装并使用 ADK，可运行：

```
pip install google-adk
```

### 1.2：配置 Gemini API Key

本 Notebook 使用 [Gemini API](https://ai.google.dev/gemini-api/docs)，需要先完成鉴权。

**1. 获取 API Key**

如果你还没有，可在 [Google AI Studio 创建 API key](https://aistudio.google.com/app/api-keys)。

**2. 将 Key 添加到 Kaggle Secrets**

接下来，你需要把 API Key 作为 Kaggle User Secret 添加到 Notebook 中。

1. 在 Notebook 编辑器顶部菜单选择 `Add-ons` -> `Secrets`。
2. 新建一个标签为 `GOOGLE_API_KEY` 的 secret。
3. 把 API Key 粘贴到 "Value" 字段并点击 "Save"。
4. 确认 `GOOGLE_API_KEY` 左侧复选框已勾选，使其附加到该 Notebook。

**3. 在 Notebook 中鉴权**

运行下面的代码单元完成鉴权。

In [1]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

✅ Gemini API key setup complete.


### 1.3：导入 ADK 组件

现在从 Agent Development Kit 和 Generative AI 库中导入接下来要用到的组件。这样可以让代码结构更清晰，并确保我们具备必要的构建模块。

In [2]:
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


### 1.4：配置重试选项

在使用 LLM 时，你可能会遇到速率限制或服务暂时不可用等瞬时错误。重试选项会通过指数退避自动重试请求，以处理这类失败。

In [3]:
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504], # Retry on these HTTP errors
)

---
## 🤔 第 2 部分：为什么要用多智能体系统？+ 你的第一个多智能体

**问题：什么都做的“全能”Agent**

单个 Agent 的能力可以很强。但当任务变复杂时会怎样？一个试图同时完成调研、写作、编辑、事实核查的“单体（monolithic）”Agent 很容易出问题。它的 instruction prompt 会变得又长又混乱；调试困难（到底哪一步失败了？）；维护成本高，结果也往往不够稳定。

**解决方案：由专家组成的团队**

与其构建一个“全能”Agent，不如构建一个**多智能体系统（multi-agent system）**。它由多个简单、专门化的 Agent 协作构成，像现实中的团队一样。每个 Agent 只做一件清晰的事（例如一个只负责调研，另一个只负责总结）。这样更易构建、更易测试，而且协同后更强大、更可靠。

想进一步了解，可参考 [ADK 中的 LLM agents 文档](https://google.github.io/adk-docs/agents/llm-agents/)。

**架构对比：单智能体 vs 多智能体团队**

<!--
```mermaid
graph TD
    subgraph Single["❌ Monolithic Agent"]
        A["One Agent Does Everything"]
    end

    subgraph Multi["✅ Multi-Agent Team"]
        B["Root Coordinator"] -- > C["Research Specialist"]
        B -- > E["Summary Specialist"]

        C -- >|findings| F["Shared State"]
        E -- >|summary| F
    end

    style A fill:#ffcccc
    style B fill:#ccffcc
    style F fill:#ffffcc
```
-->

<img width="800" src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/multi-agent-team.png" alt="Multi-agent Team" />

### 2.1 示例：调研与总结系统

我们来构建一个包含两个专用 Agent 的系统：

1. **Research Agent** - 使用 Google Search 搜索信息
2. **Summarizer Agent** - 将调研结果压缩为简明总结

In [4]:
# Research Agent: Its job is to use the google_search tool and present findings.
research_agent = Agent(
    name="ResearchAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a specialized research agent. Your only job is to use the
    google_search tool to find 2-3 pieces of relevant information on the given topic and present the findings with citations.""",
    tools=[google_search],
    output_key="research_findings",  # The result of this agent will be stored in the session state with this key.
)

print("✅ research_agent created.")

✅ research_agent created.


In [5]:
# Summarizer Agent: Its job is to summarize the text it receives.
summarizer_agent = Agent(
    name="SummarizerAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # The instruction is modified to request a bulleted list for a clear output format.
    instruction="""Read the provided research findings: {research_findings}
Create a concise summary as a bulleted list with 3-5 key points.""",
    output_key="final_summary",
)

print("✅ summarizer_agent created.")

✅ summarizer_agent created.


更多关于如何通过清晰、具体 instruction 引导 Agent 的信息，请参考 ADK 文档：[guiding agents with clear and specific instructions](https://google.github.io/adk-docs/agents/llm-agents/)。

然后，我们将这些 Agent 放到一个根 Agent（协调者）之下：

In [6]:
# Root Coordinator: Orchestrates the workflow by calling the sub-agents as tools.
root_agent = Agent(
    name="ResearchCoordinator",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # This instruction tells the root agent HOW to use its tools (which are the other agents).
    instruction="""You are a research coordinator. Your goal is to answer the user's query by orchestrating a workflow.
1. First, you MUST call the `ResearchAgent` tool to find relevant information on the topic provided by the user.
2. Next, after receiving the research findings, you MUST call the `SummarizerAgent` tool to create a concise summary.
3. Finally, present the final summary clearly to the user as your response.""",
    # We wrap the sub-agents in `AgentTool` to make them callable tools for the root agent.
    tools=[AgentTool(research_agent), AgentTool(summarizer_agent)],
)

print("✅ root_agent created.")

✅ root_agent created.


这里我们使用 `AgentTool` 包装子 Agent，使它们成为根 Agent 可调用的工具。我们会在第 2 天详细讲解 `AgentTool`。

现在运行这个 Agent，并给它一个主题：

In [7]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "What are the latest advancements in quantum computing and what do they mean for AI?"
)


 ### Created new session: debug_session_id

User > What are the latest advancements in quantum computing and what do they mean for AI?


ResearchCoordinator > The integration of quantum computing with AI, often referred to as "Quantum AI," is set to bring about a revolution. Quantum computers offer exponential speed-ups for data processing and model training, leading to faster and more efficient AI systems. They can also tackle complex problems that are currently beyond the capabilities of classical AI, with potential applications in drug discovery, materials science, and financial modeling. Furthermore, new quantum-native AI architectures and improved data processing techniques are being developed, while AI is also contributing to the advancement of quantum computing itself. This synergy is expected to drive significant industry transformation and market growth in the coming decade.


你刚刚完成了第一个多智能体系统！你使用了一个“协调者”Agent 来管理工作流，这是非常强大且灵活的模式。

‼️ 但要注意：**仅依赖 LLM instruction 去控制执行顺序，有时会出现不可预测性。** 接下来我们会看另一种模式，它能保证按步骤、确定性执行。

---

## 🚥 第 3 部分：Sequential 工作流 - 装配流水线

**问题：顺序不可控**

前面的多智能体系统能工作，但它依赖**详细 instruction prompt** 迫使 LLM 按顺序执行。这并不总是可靠。复杂 LLM 可能跳步、乱序，甚至“卡住”，导致流程不可预测。

**解决方案：固定流水线**

当你需要任务按**确定、固定顺序**执行时，可以使用 `SequentialAgent`。它像一条装配线，会严格按你给出的顺序运行子 Agent。前一个 Agent 的输出会自动成为下一个 Agent 的输入，从而形成可预测且稳定的工作流。

**适用场景：** 顺序很重要、需要线性流水线、或者每一步都依赖前一步结果。

更多信息请参考 [ADK 中的 sequential agents 文档](https://google.github.io/adk-docs/agents/workflow-agents/sequential-agents/)。

**架构示例：博客写作流水线**

<!--
```mermaid
graph LR
    A["User Input: Blog about AI"] -- > B["Outline Agent"]
    B -- >|blog_outline| C["Writer Agent"]
    C -- >|blog_draft| D["Editor Agent"]
    D -- >|final_blog| E["Output"]

    style B fill:#ffcccc
    style C fill:#ccffcc
    style D fill:#ccccff
```
-->

<img width="1000" src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/sequential-agent.png" alt="Sequential Agent" />

### 3.1 示例：用 Sequential Agents 生成博客

我们构建一个包含三个专用 Agent 的系统：

1. **Outline Agent** - 为给定主题生成博客大纲
2. **Writer Agent** - 撰写博客正文
3. **Editor Agent** - 对草稿进行结构与表达优化

In [10]:
# Outline Agent: Creates the initial blog post outline.
outline_agent = Agent(
    name="OutlineAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Create a blog outline for the given topic with:
    1. A catchy headline
    2. An introduction hook
    3. 3-5 main sections with 2-3 bullet points for each
    4. A concluding thought""",
    output_key="blog_outline",  # The result of this agent will be stored in the session state with this key.
)

print("✅ outline_agent created.")

✅ outline_agent created.


In [11]:
# Writer Agent: Writes the full blog post based on the outline from the previous agent.
writer_agent = Agent(
    name="WriterAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # The `{blog_outline}` placeholder automatically injects the state value from the previous agent's output.
    instruction="""Following this outline strictly: {blog_outline}
    Write a brief, 200 to 300-word blog post with an engaging and informative tone.""",
    output_key="blog_draft",  # The result of this agent will be stored with this key.
)

print("✅ writer_agent created.")

✅ writer_agent created.


In [12]:
# Editor Agent: Edits and polishes the draft from the writer agent.
editor_agent = Agent(
    name="EditorAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # This agent receives the `{blog_draft}` from the writer agent's output.
    instruction="""Edit this draft: {blog_draft}
    Your task is to polish the text by fixing any grammatical errors, improving the flow and sentence structure, and enhancing overall clarity.""",
    output_key="final_blog",  # This is the final output of the entire pipeline.
)

print("✅ editor_agent created.")

✅ editor_agent created.


然后把这些 Agent 放到一个 sequential agent 下，它会按列表顺序执行各个 Agent：

In [13]:
root_agent = SequentialAgent(
    name="BlogPipeline",
    sub_agents=[outline_agent, writer_agent, editor_agent],
)

print("✅ Sequential Agent created.")

google.adk.agents.sequential_agent.SequentialAgent

现在运行这个 Agent，并给它一个博客主题：

In [14]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "Write a blog post about the benefits of multi-agent systems for software developers"
)


 ### Created new session: debug_session_id

User > Write a blog post about the benefits of multi-agent systems for software developers


ResearchCoordinator > Here's a blog post about the benefits of multi-agent systems for software developers:

## Supercharge Your Development: The Power of Multi-Agent Systems

Software development is a complex, demanding field. As projects grow in scope and complexity, developers often find themselves juggling multiple responsibilities, leading to potential bottlenecks and decreased efficiency. Enter Multi-Agent Systems (MAS) – a revolutionary approach that's transforming how we build software. By leveraging specialized AI agents that collaborate to tackle challenges, MAS offers a powerful toolkit for developers.

### Key Benefits for Developers:

*   **Unleash Specialization and Modularity:** Imagine a development team where each member is an expert in a specific area – that's the essence of MAS. Each agent can focus on specialized tasks like generating features, reviewing code, or analyzing bugs. This modularity makes systems easier to maintain, test, and upgrade, and allows for the 

👏 很棒！你已经用 sequential agent 搭建了一个可靠的“装配流水线”，每一步都按可预测顺序运行。

**它非常适合前后依赖的任务，但如果任务彼此独立就会偏慢。** 接下来我们看如何让多个 Agent 同时运行以加速流程。

---
## 🛣️ 第 4 部分：Parallel 工作流 - 独立研究员

**问题：瓶颈效应**

前面的 sequential agent 很好用，但它本质是流水线。每一步都要等待前一步结束。如果你有多个**互不依赖**的任务，比如同时调研三个不同主题，按顺序执行会很慢且低效，形成不必要的等待瓶颈。

**解决方案：并发执行**

当任务相互独立时，你可以用 `ParallelAgent` 同时运行所有子 Agent。它会并发执行，显著加速流程。待并行任务全部完成后，再把结果交给最终的“汇总器（aggregator）”步骤。

**适用场景：** 任务独立、对速度敏感、可并发执行。

更多信息请参考 [ADK 中的 parallel agents 文档](https://google.github.io/adk-docs/agents/workflow-agents/parallel-agents/)。

**架构示例：多主题调研**

<!--
```mermaid
graph TD
    A["User Request: Research 3 topics"] -- > B["Parallel Execution"]
    B -- > C["Tech Researcher"]
    B -- > D["Health Researcher"]
    B -- > E["Finance Researcher"]

    C -- > F["Aggregator"]
    D -- > F
    E -- > F
    F -- > G["Combined Report"]

    style B fill:#ffffcc
    style F fill:#ffccff
```
-->

<img width="600" src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/parallel-agent.png" alt="Parallel Agent" />

### 4.1 示例：并行多主题调研

我们构建一个包含四个 Agent 的系统：

1. **Tech Researcher** - 调研 AI/ML 新闻与趋势
2. **Health Researcher** - 调研医疗最新进展与趋势
3. **Finance Researcher** - 调研金融与金融科技趋势
4. **Aggregator Agent** - 将三路调研结果汇总为一个总结

In [15]:
# Tech Researcher: Focuses on AI and ML trends.
tech_researcher = Agent(
    name="TechResearcher",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research the latest AI/ML trends. Include 3 key developments,
the main companies involved, and the potential impact. Keep the report very concise (100 words).""",
    tools=[google_search],
    output_key="tech_research",  # The result of this agent will be stored in the session state with this key.
)

print("✅ tech_researcher created.")

✅ tech_researcher created.


In [16]:
# Health Researcher: Focuses on medical breakthroughs.
health_researcher = Agent(
    name="HealthResearcher",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research recent medical breakthroughs. Include 3 significant advances,
their practical applications, and estimated timelines. Keep the report concise (100 words).""",
    tools=[google_search],
    output_key="health_research",  # The result will be stored with this key.
)

print("✅ health_researcher created.")

✅ health_researcher created.


In [17]:
# Finance Researcher: Focuses on fintech trends.
finance_researcher = Agent(
    name="FinanceResearcher",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research current fintech trends. Include 3 key trends,
their market implications, and the future outlook. Keep the report concise (100 words).""",
    tools=[google_search],
    output_key="finance_research",  # The result will be stored with this key.
)

print("✅ finance_researcher created.")

✅ finance_researcher created.


In [18]:
# The AggregatorAgent runs *after* the parallel step to synthesize the results.
aggregator_agent = Agent(
    name="AggregatorAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # It uses placeholders to inject the outputs from the parallel agents, which are now in the session state.
    instruction="""Combine these three research findings into a single executive summary:

    **Technology Trends:**
    {tech_research}
    
    **Health Breakthroughs:**
    {health_research}
    
    **Finance Innovations:**
    {finance_research}
    
    Your summary should highlight common themes, surprising connections, and the most important key takeaways from all three reports. The final summary should be around 200 words.""",
    output_key="executive_summary",  # This will be the final output of the entire system.
)

print("✅ aggregator_agent created.")

✅ aggregator_agent created.


👉 **然后把这些 Agent 组合到一个 parallel agent 中，并将其嵌套进 sequential agent。**

这样设计可以保证：先并行执行调研 Agent；待全部调研完成后，再由 aggregator Agent 把结果汇总成单份报告：

In [19]:
# The ParallelAgent runs all its sub-agents simultaneously.
parallel_research_team = ParallelAgent(
    name="ParallelResearchTeam",
    sub_agents=[tech_researcher, health_researcher, finance_researcher],
)

# This SequentialAgent defines the high-level workflow: run the parallel team first, then run the aggregator.
root_agent = SequentialAgent(
    name="ResearchSystem",
    sub_agents=[parallel_research_team, aggregator_agent],
)

print("✅ Parallel and Sequential Agents created.")

✅ Parallel and Sequential Agents created.


现在运行这个 Agent，并给它一个调研这些主题的 Prompt：

In [20]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "Run the daily executive briefing on Tech, Health, and Finance"
)


 ### Created new session: debug_session_id

User > Run the daily executive briefing on Tech, Health, and Finance
TechResearcher > **Key AI/ML Trends in 2026:**

1.  **Generative AI Expansion:** Generative AI is moving beyond text to create diverse content like graphics, video, and music. Companies like **Google** (Muse, Imagen) and **Runway ML** are at the forefront, with applications projected to grow significantly.
2.  **Shift to Small Language Models (SLMs):** SLMs offer efficient AI capabilities, suitable for devices like smartphones and edge computing. This trend makes AI more accessible and cost-effective for businesses.
3.  **AI-Powered Automation and Agentic AI:** AI is increasingly automating complex tasks and processes, moving towards "agentic AI" that can complete tasks with minimal human intervention. This drives efficiency and cost reduction across industries.

**Main Companies Involved:**
Leading companies include **Google**, **Microsoft**, **NVIDIA**, **Amazon**, and **

🎉 很好！你已经看到并行 Agent 如何通过并发执行独立任务，显著加快工作流。

到目前为止，我们的工作流都是从头到尾跑完就结束。**但如果你需要反复审阅并改进输出怎么办？** 接下来我们会构建一个能循环迭代、持续优化结果的工作流。

---
## ➰ 第 5 部分：Loop 工作流 - 迭代打磨循环

**问题：一次生成的质量上限**

我们目前看到的工作流都是从开始到结束执行一次。`SequentialAgent` 和 `ParallelAgent` 产出最终结果后就停止。这种“一次成稿”的方式不适合需要反复优化与质量控制的任务。如果故事初稿很差，我们就没有机制去审阅并要求重写。

**解决方案：迭代式优化**

当任务需要通过“反馈-修订”循环来持续改进时，可以使用 `LoopAgent`。`LoopAgent` 会反复执行一组子 Agent，*直到满足特定条件，或达到最大迭代次数*。这样就形成了打磨循环，让系统可以不断优化自己的输出。

**适用场景：** 需要迭代改进、重视质量精修、或必须多轮循环。

更多信息请参考 [ADK 中的 loop agents 文档](https://google.github.io/adk-docs/agents/workflow-agents/loop-agents/)。

**架构示例：故事写作与评审循环**

<!--
```mermaid
graph TD
    A["Initial Prompt"] -- > B["Writer Agent"]
    B -- >|story| C["Critic Agent"]
    C -- >|critique| D{"Iteration < Max<br>AND<br>Not Approved?"}
    D -- >|Yes| B
    D -- >|No| E["Final Story"]

    style B fill:#ccffcc
    style C fill:#ffcccc
    style D fill:#ffffcc
```
-->

<img width="250" src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/loop-agent.png" alt="Loop Agent" />

### 5.1 示例：迭代式故事优化

我们构建一个包含两个 Agent 的系统：

1. **Writer Agent** - 写出短故事草稿
2. **Critic Agent** - 审阅并提出可执行的改进建议

In [21]:
# This agent runs ONCE at the beginning to create the first draft.
initial_writer_agent = Agent(
    name="InitialWriterAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Based on the user's prompt, write the first draft of a short story (around 100-150 words).
    Output only the story text, with no introduction or explanation.""",
    output_key="current_story",  # Stores the first draft in the state.
)

print("✅ initial_writer_agent created.")

✅ initial_writer_agent created.


In [22]:
# This agent's only job is to provide feedback or the approval signal. It has no tools.
critic_agent = Agent(
    name="CriticAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a constructive story critic. Review the story provided below.
    Story: {current_story}
    
    Evaluate the story's plot, characters, and pacing.
    - If the story is well-written and complete, you MUST respond with the exact phrase: "APPROVED"
    - Otherwise, provide 2-3 specific, actionable suggestions for improvement.""",
    output_key="critique",  # Stores the feedback in the state.
)

print("✅ critic_agent created.")

✅ critic_agent created.


现在我们需要一种机制，让循环能够根据 Critic 的反馈真正“停下来”。`LoopAgent` 本身并不会自动知道 "APPROVED" 就表示“停止”。

我们需要一个 Agent 给出明确的终止信号。

这个机制分两步：

1. 定义一个简单 Python 函数，作为 `LoopAgent` 可识别的“退出”信号。
2. 创建一个可以在满足条件时调用该函数的 Agent。

首先，定义 `exit_loop` 函数：

In [24]:
# This is the function that the RefinerAgent will call to exit the loop.
def exit_loop():
    """Call this function ONLY when the critique is 'APPROVED', indicating the story is finished and no more changes are needed."""
    return {"status": "approved", "message": "Story approved. Exiting refinement loop."}


print("✅ exit_loop function created.")

✅ exit_loop function created.


为了让 Agent 能调用这个 Python 函数，我们先用 `FunctionTool` 包装它。然后创建带有该工具的 `RefinerAgent`。

👉 **注意它的 instruction：** 这个 Agent 是循环里的“决策大脑”。它会读取 `CriticAgent` 生成的 `{critique}`，并决定是 (1) 调用 `exit_loop` 工具，还是 (2) 重写故事。

In [25]:
# This agent refines the story based on critique OR calls the exit_loop function.
refiner_agent = Agent(
    name="RefinerAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a story refiner. You have a story draft and critique.
    
    Story Draft: {current_story}
    Critique: {critique}
    
    Your task is to analyze the critique.
    - IF the critique is EXACTLY "APPROVED", you MUST call the `exit_loop` function and nothing else.
    - OTHERWISE, rewrite the story draft to fully incorporate the feedback from the critique.""",
    output_key="current_story",  # It overwrites the story with the new, refined version.
    tools=[
        FunctionTool(exit_loop)
    ],  # The tool is now correctly initialized with the function reference.
)

print("✅ refiner_agent created.")

✅ refiner_agent created.


然后把这些 Agent 放到 loop agent 下，并将该 loop agent 嵌套在 sequential agent 中。

这样的设计可以保证系统先生成初稿，再运行精修循环，最多执行到你设置的 `max_iterations` 次：

In [26]:
# The LoopAgent contains the agents that will run repeatedly: Critic -> Refiner.
story_refinement_loop = LoopAgent(
    name="StoryRefinementLoop",
    sub_agents=[critic_agent, refiner_agent],
    max_iterations=2,  # Prevents infinite loops
)

# The root agent is a SequentialAgent that defines the overall workflow: Initial Write -> Refinement Loop.
root_agent = SequentialAgent(
    name="StoryPipeline",
    sub_agents=[initial_writer_agent, story_refinement_loop],
)

print("✅ Loop and Sequential Agents created.")

✅ Loop and Sequential Agents created.


现在运行这个 Agent，并给它一个短故事主题：

In [27]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "Write a short story about a lighthouse keeper who discovers a mysterious, glowing map"
)


 ### Created new session: debug_session_id

User > Write a short story about a lighthouse keeper who discovers a mysterious, glowing map
InitialWriterAgent > Elias polished the Fresnel lens, the rhythmic sweep of light a familiar comfort against the encroaching dusk. For twenty years, the lighthouse had been his world, a solitary sentinel against the churning sea. Tonight, however, something was different. Tucked within a rotting sea chest, unearthed during a rare tidying spree, he found it: a map, not of any coast he knew, but inked on parchment that shimmered with an inner, phosphorescent glow. Strange symbols, like constellations he’d never seen, pulsed with a soft, ethereal light. He traced a finger over a swirling vortex, a shiver running down his spine. The sea outside, usually a predictable roar, seemed to hold its breath, listening. This was no ordinary chart; it was a whisper from a world beyond the horizon.
CriticAgent > This is a strong opening that effectively establishes 

现在你已经实现了 loop agent，构建出一个可以反复审阅并优化自身输出的高级系统。这是保障高质量结果的关键模式。

至此你已经拥有完整的工作流模式工具箱。下面我们把它们汇总起来，看看如何为具体场景选择合适模式。

--- 
## 第 6 部分：总结 - 如何选择合适模式

### 决策树：该选哪种工作流模式？

<!--
```mermaid
graph TD
    A{"What kind of workflow do you need?"} -- > B["Fixed Pipeline<br>(A → B → C)"];
    A -- > C["Concurrent Tasks<br>(Run A, B, C all at once)"];
    A -- > D["Iterative Refinement<br>(A ⇆ B)"];
    A -- > E["Dynamic Decisions<br>(Let the LLM decide what to do)"];

    B -- > B_S["Use <b>SequentialAgent</b>"];
    C -- > C_S["Use <b>ParallelAgent</b>"];
    D -- > D_S["Use <b>LoopAgent</b>"];
    E -- > E_S["Use <b>LLM Orchestrator</b><br>(Agent with other agents as tools)"];

    style B_S fill:#f9f,stroke:#333,stroke-width:2px
    style C_S fill:#ccf,stroke:#333,stroke-width:2px
    style D_S fill:#cff,stroke:#333,stroke-width:2px
    style E_S fill:#cfc,stroke:#333,stroke-width:2px
```
-->

<img width="1000" src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/agent-decision-tree.png" alt="Agent Decision Tree" />

### 快速对照表

| 模式 | 适用场景 | 示例 | 核心特性 |
|---------|-------------|---------|-------------|
| **基于 LLM（sub_agents）** | 需要动态编排 | 调研 + 总结 | 由 LLM 决定调用路径 |
| **Sequential** | 顺序重要、线性流水线 | 大纲 → 写作 → 编辑 | 确定性顺序 |
| **Parallel** | 任务独立、重视速度 | 多主题调研 | 并发执行 |
| **Loop** | 需要迭代改进 | Writer + Critic 精修 | 重复循环 |

---

## ✅ 恭喜！你已经成为 Agent 编排者

在本 Notebook 中，你完成了从单智能体到**多智能体系统**的关键跃迁。

你看到了为什么“专家团队”比“全能单体 Agent”更容易构建、调试与维护。更重要的是，你学会了如何担任这个团队的**导演**。

你使用 `SequentialAgent`、`ParallelAgent` 和 `LoopAgent` 构建了确定性工作流，也用 LLM 作为“管理者”实现动态决策。同时你还掌握了多智能体协作的“管道能力”：用 `output_key` 在 Agent 之间传递状态。

**ℹ️ 说明：无需提交！**

这个 Notebook 仅用于动手实践与学习。完成课程**不需要**把它提交到任何地方。

### 📚 继续学习

你可以参考以下文档深入了解：

- [Agents in ADK](https://google.github.io/adk-docs/agents/)
- [Sequential Agents in ADK](https://google.github.io/adk-docs/agents/workflow-agents/sequential-agents/)
- [Parallel Agents in ADK](https://google.github.io/adk-docs/agents/workflow-agents/parallel-agents/)
- [Loop Agents in ADK](https://google.github.io/adk-docs/agents/workflow-agents/loop-agents/)
- [Custom Agents in ADK](https://google.github.io/adk-docs/agents/custom-agents/)

### 🎯 下一步

准备迎接下一个挑战了吗？继续关注 Day 2 Notebook，我们将学习如何创建 **Custom Functions、使用 MCP Tools**，以及管理**长时间运行任务**！